In [2]:
from multiprocessing import Pool, TimeoutError
import time
import os

def f(x):
    return x*x

if __name__ == '__main__':
    # start 4 worker processes
    with Pool(processes=4) as pool:

        # print "[0, 1, 4,..., 81]"
        print(pool.map(f, range(10)))

        # print same numbers in arbitrary order
        for i in pool.imap_unordered(f, range(10)):
            print(i)

        # evaluate "f(20)" asynchronously
        res = pool.apply_async(f, (20,))      # runs in *only* one process
        print(res.get(timeout=1))             # prints "400"

        # evaluate "os.getpid()" asynchronously
        res = pool.apply_async(os.getpid, ()) # runs in *only* one process
        print(res.get(timeout=1))             # prints the PID of that process

        # launching multiple evaluations asynchronously *may* use more processes
        multiple_results = [pool.apply_async(os.getpid, ()) for i in range(4)]
        print([res.get(timeout=1) for res in multiple_results])

        # make a single worker sleep for 10 seconds
        res = pool.apply_async(time.sleep, (10,))
        try:
            print(res.get(timeout=1))
        except TimeoutError:
            print("We lacked patience and got a multiprocessing.TimeoutError")

        print("For the moment, the pool remains available for more work")

    # exiting the 'with'-block has stopped the pool
    print("Now the pool is closed and no longer available")

[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]
0
1
4
9
16
25
36
49
64
81
400
4070425
[4070423, 4070423, 4070423, 4070423]
We lacked patience and got a multiprocessing.TimeoutError
For the moment, the pool remains available for more work
Now the pool is closed and no longer available


In [7]:
import multiprocessing as mp
import time

def task(n):
    time.sleep(2)  # 模拟耗时操作
    print(f'Task {n} completed')

def callback(result):
    # 这个回调函数可以在任务完成时进行额外处理工作
    pass

def main():
    total_tasks = 45
    max_processes = 15
    
    # 创建一个进程池
    pool = mp.Pool(processes=max_processes)
    
    # 用于跟踪提交的任务数
    submitted_tasks = 0

    # 初始提交15个任务
    for i in range(max_processes):
        pool.apply_async(task, args=(i,), callback=callback)
        submitted_tasks += 1
    
    # 使用队列或其他机制继续补充
    while submitted_tasks < total_tasks:
        # wait if needed
        pool.apply_async(task, args=(submitted_tasks,), callback=callback)
        submitted_tasks += 1

    pool.close()  # 不再接收新任务
    pool.join()   # 等待所有任务完成
    
if __name__ == '__main__':
    main()


Task 1 completedTask 13 completedTask 12 completedTask 11 completedTask 5 completedTask 2 completedTask 6 completedTask 9 completedTask 4 completedTask 10 completed



Task 0 completedTask 7 completedTask 14 completedTask 3 completed





Task 8 completed




Task 16 completed
Task 15 completed
Task 18 completedTask 17 completed

Task 19 completedTask 22 completedTask 21 completedTask 20 completed

Task 23 completedTask 24 completed

Task 26 completedTask 27 completed
Task 28 completed


Task 29 completedTask 25 completed


Task 30 completed
Task 31 completed
Task 32 completed
Task 33 completed
Task 34 completed
Task 35 completed
Task 37 completedTask 36 completedTask 38 completedTask 41 completedTask 40 completed
Task 42 completed
Task 43 completed

Task 44 completedTask 39 completed






In [4]:
for i in range(4,15):
    print(i)

4
5
6
7
8
9
10
11
12
13
14


In [1]:
import multiprocessing
import time

def task(i):
    print(f"Task {i} started.")
    time.sleep(i)  # 模拟不同任务的执行时间
    print(f"Task {i} finished.")

if __name__ == "__main__":
    with multiprocessing.Pool(processes=3) as pool:
        results = []
        for i in range(6):
            result = pool.apply_async(task, args=(i,))
            results.append(result)

        while results:
            for i, result in enumerate(results):
                try:
                    result.get(timeout=5)  # 设置超时时间
                    results.pop(i)  # 从列表中移除已完成的任务
                    print(f"Task {i} finished.")
                except multiprocessing.TimeoutError:
                    print(f"Task {i} timed out.")

Task 1 started.Task 0 started.Task 2 started.

Task 0 finished.

Task 3 started.
Task 0 finished.
Task 1 finished.
Task 4 started.
Task 2 finished.
Task 5 started.
Task 1 finished.
Task 3 finished.
Task 4 finished.
Task 2 finished.
Task 0 finished.
Task 5 finished.
Task 1 finished.
Task 0 finished.


In [2]:
import multiprocessing
import time
import os

# 示例任务函数，模拟执行时间不同的任务
def task(name, sleep_time):
    print(f"Process {name} started with PID {os.getpid()}, will run for {sleep_time} seconds.")
    time.sleep(sleep_time)  # 模拟任务执行
    print(f"Process {name} completed.")
    return name

def monitor_task(result, timeout):
    """监控任务的执行，如果超时则终止任务"""
    try:
        result.get(timeout=timeout)  # 等待获取任务结果，并设置超时时间
    except multiprocessing.TimeoutError:
        print(f"Task exceeded the time limit of {timeout} seconds and was terminated.")

def run_tasks_with_timeout(tasks, max_processes=3, timeout=5):
    # 创建进程池，设置最大并发进程数
    pool = multiprocessing.Pool(processes=max_processes)
    results = []

    # 提交任务到进程池，并异步监控
    for task_args in tasks:
        result = pool.apply_async(task, task_args)  # 异步提交任务
        monitor = multiprocessing.Process(target=monitor_task, args=(result, timeout))
        monitor.start()  # 启动监控进程
        results.append(monitor)

    # 等待所有监控任务完成
    for monitor in results:
        monitor.join()

    pool.close()
    pool.join()

# 守护进程函数，负责管理任务的执行
def daemon_process():
    print(f"Daemon process started with PID {os.getpid()}")
    # 要运行的任务列表，每个任务包含名称和睡眠时间
    tasks = [
        ("Task1", 3),
        ("Task2", 6),
        ("Task3", 1),
        ("Task4", 8),
        ("Task5", 4),
        ("Task6", 3),
    ]
    # 运行带有超时管理的任务池
    run_tasks_with_timeout(tasks)

if __name__ == "__main__":
    # 创建守护进程
    daemon = multiprocessing.Process(target=daemon_process)
    daemon.daemon = True  # 设置为守护进程
    daemon.start()  # 启动守护进程

    # 主进程等待守护进程结束
    daemon.join()


Daemon process started with PID 16289


Process Process-4:
Traceback (most recent call last):
  File "/home/dy/Softwares/miniconda3/envs/Vit/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/dy/Softwares/miniconda3/envs/Vit/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_16074/1968909934.py", line 51, in daemon_process
    run_tasks_with_timeout(tasks)
  File "/tmp/ipykernel_16074/1968909934.py", line 21, in run_tasks_with_timeout
    pool = multiprocessing.Pool(processes=max_processes)
  File "/home/dy/Softwares/miniconda3/envs/Vit/lib/python3.10/multiprocessing/context.py", line 119, in Pool
    return Pool(processes, initializer, initargs, maxtasksperchild,
  File "/home/dy/Softwares/miniconda3/envs/Vit/lib/python3.10/multiprocessing/pool.py", line 215, in __init__
    self._repopulate_pool()
  File "/home/dy/Softwares/miniconda3/envs/Vit/lib/python3.10/multiprocessing/pool.py", line 306, in _re

In [ ]:
# 执行函数
from functools import partial
from multiprocessing.pool import ThreadPool


# 示例任务函数，模拟执行时间不同的任务
def task(name, sleep_time):
    print(f"Process {name} started with PID {os.getpid()}, will run for {sleep_time} seconds.")
    time.sleep(sleep_time)  # 模拟任务执行
    print(f"Process {name} completed.")
    return name

# 守护进程
def abortable_check(self, func, *args, **kwargs):
    timeout = kwargs.get('timeout', None)
    p = ThreadPool(1)
    res = p.apply_async(func, args=args)
    try:
        out = res.get(timeout)  # Wait timeout seconds for func to complete.
        # print(out)
        return out
    except multiprocessing.TimeoutError:
        print("{} timeout".format(args[0]))
        return (args[0], 444)

# 回调函数
def collectMyResult(self, result):
    if result is not None and result[1] == 404:
        self.brokenlinks.append(result[0])
    if result is not None and result[1] == 444:
        self.timeout.append(result[0])
    self.count += 1 #全局变量统计任务执行进度
    print('\r进度：{}/{}'.format(self.count,self.linkslen), end ='')


def main():
    pool = multiprocessing.Pool(maxtasksperchild=10)
    for l in links:
        a = [l]
        abortable_func = partial(self.abortable_check, self.check, timeout=5)
        pool.apply_async(abortable_func, args=a, callback=self.collectMyResult)


In [5]:
import multiprocessing
import time
import os

# 示例任务函数，模拟执行时间不同的任务
def task(name, sleep_time):
    print(f"Process {name} started with PID {os.getpid()}, will run for {sleep_time} seconds.")
    time.sleep(sleep_time)  # 模拟任务执行
    print(f"Process {name} completed.")
    return name

def monitor_task(result, timeout):
    """监控任务的执行，如果超时则终止任务"""
    try:
        result.get(timeout=timeout)  # 等待获取任务结果，并设置超时时间
    except multiprocessing.TimeoutError:
        print(f"Task exceeded the time limit of {timeout} seconds and was terminated.")

def run_tasks_with_timeout(tasks, max_processes=3, timeout=5):
    # 创建进程池，设置最大并发进程数
    pool = multiprocessing.Pool(processes=max_processes)
    results = []

    # 提交任务到进程池，并异步监控
    for task_args in tasks:
        result = pool.apply_async(task, task_args)  # 异步提交任务
        monitor = multiprocessing.Process(target=monitor_task, args=(result, timeout))
        monitor.start()  # 启动监控进程
        results.append(monitor)

    # 等待所有监控任务完成
    for monitor in results:
        monitor.join()

    pool.close()
    pool.join()

# 守护进程函数，负责管理任务的执行
def daemon_process():
    print(f"Daemon process started with PID {os.getpid()}")
    # 要运行的任务列表，每个任务包含名称和睡眠时间
    tasks = [
        ("Task1", 3),
        ("Task2", 6),
        ("Task3", 1),
        ("Task4", 8),
        ("Task5", 4),
        ("Task6", 3),
    ]
    # 运行带有超时管理的任务池
    run_tasks_with_timeout(tasks)

if __name__ == "__main__":
    # 创建守护进程
    daemon = multiprocessing.Process(target=daemon_process)
    daemon.daemon = True  # 设置为守护进程
    daemon.start()  # 启动守护进程

    # 主进程等待守护进程结束
    daemon.join()


Daemon process started with PID 16612

Process Process-9:
Traceback (most recent call last):
  File "/home/dy/Softwares/miniconda3/envs/Vit/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/dy/Softwares/miniconda3/envs/Vit/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_16074/1968909934.py", line 51, in daemon_process
    run_tasks_with_timeout(tasks)
  File "/tmp/ipykernel_16074/1968909934.py", line 21, in run_tasks_with_timeout
    pool = multiprocessing.Pool(processes=max_processes)
  File "/home/dy/Softwares/miniconda3/envs/Vit/lib/python3.10/multiprocessing/context.py", line 119, in Pool
    return Pool(processes, initializer, initargs, maxtasksperchild,
  File "/home/dy/Softwares/miniconda3/envs/Vit/lib/python3.10/multiprocessing/pool.py", line 215, in __init__
    self._repopulate_pool()
  File "/home/dy/Softwares/miniconda3/envs/Vit/lib/python3.10/multiprocessing/pool.py", line 306, in _re

In [7]:
import multiprocessing
import time
import os

# 示例任务函数，模拟执行时间不同的任务
def task(name, sleep_time):
    print(f"Process {name} started with PID {os.getpid()}, will run for {sleep_time} seconds.")
    time.sleep(sleep_time)  # 模拟任务执行
    print(f"Process {name} completed.")
    return name

def check_task_timeout(result, start_time, timeout):
    """检查任务是否超时，并终止超时任务"""
    try:
        # 计算任务已运行的时间
        elapsed_time = time.time() - start_time
        if elapsed_time >= timeout:
            raise multiprocessing.TimeoutError
        return result.get(timeout=timeout - elapsed_time)  # 等待剩余的时间来获取结果
    except multiprocessing.TimeoutError:
        print(f"Task exceeded the time limit of {timeout} seconds and was terminated.")
        return None

def run_tasks_with_timeout(tasks, max_processes=3, timeout=5):
    # 创建进程池，设置最大并发进程数
    pool = multiprocessing.Pool(processes=max_processes)
    results = []
    start_times = []

    # 提交任务到进程池，并记录提交时间
    for task_args in tasks:
        result = pool.apply_async(task, task_args)  # 异步提交任务
        results.append(result)
        start_times.append(time.time())  # 记录提交任务的时间

    # 每分钟检查任务是否超时
    while any([not result.ready() for result in results]):  # 如果有任务没有完成，则继续监控
        time.sleep(3)  # 等待1分钟

        for i, result in enumerate(results):
            if not result.ready():  # 检查未完成的任务
                elapsed_time = time.time() - start_times[i]
                if elapsed_time > timeout:  # 检查任务是否已经超时
                    print(f"Task {i+1} has timed out after {elapsed_time} seconds.")
                    result._cache.clear()  # 清除缓存，模拟终止进程

    pool.close()
    pool.join()

def main():
    # 要运行的任务列表，每个任务包含名称和睡眠时间
    tasks = [
        ("Task1", 3),
        ("Task2", 6),
        ("Task3", 1),
        ("Task4", 8),
        ("Task5", 4),
        ("Task6", 3),
    ]
    # 运行带有超时管理的任务池
    run_tasks_with_timeout(tasks, max_processes=3, timeout=5)

if __name__ == "__main__":
    main()


Process Task3 started with PID 16826, will run for 1 seconds.Process Task1 started with PID 16824, will run for 3 seconds.Process Task2 started with PID 16825, will run for 6 seconds.


Process Task3 completed.
Process Task4 started with PID 16826, will run for 8 seconds.
Process Task1 completed.
Process Task5 started with PID 16824, will run for 4 seconds.
Process Task2 completed.
Process Task6 started with PID 16825, will run for 3 seconds.
Task 2 has timed out after 6.00437331199646 seconds.
Task 4 has timed out after 6.004705905914307 seconds.
Task 5 has timed out after 6.0047314167022705 seconds.
Task 6 has timed out after 6.004735946655273 seconds.
Process Task5 completed.
Process Task6 completed.Process Task4 completed.

Task 2 has timed out after 9.006781101226807 seconds.
Task 4 has timed out after 9.006917476654053 seconds.
Task 5 has timed out after 9.00691294670105 seconds.
Task 6 has timed out after 9.006935119628906 seconds.
Task 2 has timed out after 12.00851583480835 se

KeyboardInterrupt: 

In [8]:
import multiprocessing
import time
import os
import psutil

def task(i):
    print(f"Task {i} started.")
    time.sleep(i * 2)  # 模拟不同任务执行时间
    print(f"Task {i} finished.")

def monitor_and_kill(pids, start_times, timeout=60):
    while pids:
        for i, pid in enumerate(pids):
            if psutil.pid_exists(pid):
                elapsed_time = time.time() - start_times[i]
                if elapsed_time > timeout:
                    os.kill(pid, 9)
                    print(f"Task {i} timed out and killed.")
                    pids.pop(i)
                    start_times.pop(i)
                    break
        time.sleep(60)  # 每分钟检查一次

if __name__ == "__main__":
    with multiprocessing.Pool(processes=3) as pool:
        pids = []
        start_times = []
        for i in range(6):
            result = pool.apply_async(task, args=(i,))
            pids.append(result._process.pid)
            start_times.append(time.time())

        monitor_and_kill(pids, start_times)

AttributeError: 'ApplyResult' object has no attribute '_process'

In [9]:
import multiprocessing
import time
import os
import signal

def worker(job_id):
    """Worker function that simulates a long-running task."""
    print(f"Process {os.getpid()} with job_id {job_id} started.")
    time.sleep(10)  # Simulate long-running task
    return f"Result from job_id {job_id}"

def check_processes(pool, job_ids, start_times):
    """Check the status of processes and kill those that are timed out."""
    while True:
        time.sleep(60)  # Check every minute
        for job_id, start_time in zip(job_ids, start_times):
            elapsed_time = time.time() - start_time
            if elapsed_time > 60:  # Set timeout to 60 seconds
                print(f"Killing process with job_id {job_id}")
                os.kill(job_id, signal.SIGTERM)

if __name__ == "__main__":
    num_processes = 6
    max_processes = 3

    with multiprocessing.Pool(processes=max_processes) as pool:
        job_ids = []
        start_times = []
        
        # Submit jobs to the process pool
        for i in range(num_processes):
            process = pool.apply_async(worker, (i,))
            job_ids.append(process._get_pid())  # Save the PID of the process
            start_times.append(time.time())  # Save the start time

        # Start a separate thread to check the processes
        check_processes(pool, job_ids, start_times)

        # Wait for all processes to complete
        pool.close()
        pool.join()


AttributeError: 'ApplyResult' object has no attribute '_get_pid'